# Lab 08 — Map Visualization (Solution, nâng cấp)

Lab này bao gồm: **projection**, **choropleth**, **cartogram**, **cartogram heatmap**, **contour map**, **flow map**.

## 0) Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

root = Path("..").resolve().parent
df = pd.read_csv(root / "data" / "gapminder.csv")
d2007 = df[df["year"] == 2007].copy()
d2007.head()

## 1) Projection examples (phép chiếu khác nhau)

In [ ]:
sample = d2007[d2007["continent"].isin(["Asia", "Europe", "Africa"])].copy()
projections = ["equirectangular", "natural earth", "orthographic", "mercator"]

for p in projections:
    fig = px.scatter_geo(
        sample,
        locations="iso_alpha",
        color="continent",
        size="pop",
        hover_name="country",
        projection=p,
        title=f"Projection: {p}",
    )
    fig.show()

## 2) Choropleth map

In [ ]:
fig_choro = px.choropleth(
    d2007,
    locations="iso_alpha",
    color="lifeExp",
    hover_name="country",
    color_continuous_scale="Viridis",
    projection="natural earth",
    title="Choropleth: Life expectancy (2007)",
)
fig_choro.show()

## 3) Cartogram (Dorling-style bubble cartogram approximation)

In [ ]:
# Dùng một tập quốc gia đại diện với centroid gần đúng
carto = pd.DataFrame({
    "country": ["United States", "China", "India", "Brazil", "Nigeria", "France", "Australia", "Russia"],
    "iso_alpha": ["USA", "CHN", "IND", "BRA", "NGA", "FRA", "AUS", "RUS"],
    "lat": [39.8, 35.9, 22.6, -14.2, 9.1, 46.2, -25.3, 61.5],
    "lon": [-98.6, 104.2, 79.0, -51.9, 8.7, 2.2, 133.8, 105.3],
})

carto = carto.merge(d2007[["iso_alpha", "pop", "gdpPercap", "lifeExp"]], on="iso_alpha", how="left")
# Tránh lỗi marker size NaN khi một vài mã quốc gia không map được
d2007_pop_median = d2007["pop"].median()
d2007_life_median = d2007["lifeExp"].median()
carto["pop"] = carto["pop"].fillna(d2007_pop_median)
carto["lifeExp"] = carto["lifeExp"].fillna(d2007_life_median)
carto = carto.dropna(subset=["lat", "lon"])

fig_carto = px.scatter_geo(
    carto,
    lat="lat",
    lon="lon",
    size="pop",
    color="lifeExp",
    hover_name="country",
    projection="natural earth",
    size_max=60,
    title="Cartogram approximation (bubble size = population)",
)
fig_carto.show()

## 4) Cartogram heatmap (tile cartogram)

In [ ]:
# Grid tile cartogram: mọi tile diện tích bằng nhau, màu biểu diễn giá trị
tile = pd.DataFrame({
    "country": ["USA", "CHN", "IND", "BRA", "NGA", "FRA", "AUS", "RUS"],
    "x": [0, 1, 2, 0, 1, 2, 0, 1],
    "y": [0, 0, 0, 1, 1, 1, 2, 2],
    "iso_alpha": ["USA", "CHN", "IND", "BRA", "NGA", "FRA", "AUS", "RUS"],
})
tile = tile.merge(d2007[["iso_alpha", "lifeExp"]], on="iso_alpha", how="left")

pivot_tile = tile.pivot(index="y", columns="x", values="lifeExp")
plt.figure(figsize=(6, 4))
sns.heatmap(pivot_tile, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={"label": "Life Expectancy"})
plt.title("Cartogram heatmap (equal-size tiles)")
plt.xlabel("Tile X"); plt.ylabel("Tile Y")
plt.tight_layout(); plt.show()
tile

## 5) Contour map (isoline over lat-lon field)

In [ ]:
# Tạo trường scalar giả lập trên lat-lon để minh họa contour
lon = np.linspace(-180, 180, 121)
lat = np.linspace(-60, 80, 71)
LON, LAT = np.meshgrid(lon, lat)

# Hàm giả lập (ví dụ “temperature-like field”)
Z = (
    20 + 10 * np.sin(np.deg2rad(LAT))
    + 6 * np.cos(np.deg2rad(LON / 2))
    - 0.02 * (LAT - 10) ** 2
)

fig_contour = go.Figure(data=go.Contour(
    x=lon, y=lat, z=Z,
    colorscale="RdYlBu_r",
    contours=dict(showlabels=True),
    colorbar=dict(title="Value"),
))
fig_contour.update_layout(
    title="Contour map (synthetic field on lat-lon)",
    xaxis_title="Longitude", yaxis_title="Latitude",
)
fig_contour.show()

## 6) Flow map (OD lines)

In [ ]:
flows = pd.DataFrame({
    "src": ["Singapore", "Dubai", "London", "New York", "Tokyo"],
    "src_lat": [1.35, 25.20, 51.50, 40.71, 35.68],
    "src_lon": [103.82, 55.27, -0.12, -74.00, 139.69],
    "dst": ["London", "New York", "Dubai", "Tokyo", "Singapore"],
    "dst_lat": [51.50, 40.71, 25.20, 35.68, 1.35],
    "dst_lon": [-0.12, -74.00, 55.27, 139.69, 103.82],
    "flow": [80, 65, 40, 55, 35],
})

fig_flow = go.Figure()
for _, r in flows.iterrows():
    fig_flow.add_trace(
        go.Scattergeo(
            lon=[r["src_lon"], r["dst_lon"]],
            lat=[r["src_lat"], r["dst_lat"]],
            mode="lines",
            line=dict(width=1 + r["flow"] / 25, color="royalblue"),
            opacity=0.7,
            hoverinfo="text",
            text=f"{r['src']} -> {r['dst']}: {r['flow']}",
            showlegend=False,
        )
    )

# Nút nguồn/đích
cities = pd.DataFrame({
    "city": pd.unique(flows[["src", "dst"]].values.ravel("K"))
})
city_lookup = {}
for _, r in flows.iterrows():
    city_lookup[r["src"]] = (r["src_lat"], r["src_lon"])
    city_lookup[r["dst"]] = (r["dst_lat"], r["dst_lon"])
cities["lat"] = cities["city"].map(lambda c: city_lookup[c][0])
cities["lon"] = cities["city"].map(lambda c: city_lookup[c][1])

fig_flow.add_trace(go.Scattergeo(
    lon=cities["lon"], lat=cities["lat"], mode="markers+text",
    text=cities["city"], textposition="top center",
    marker=dict(size=6, color="crimson"),
    name="Cities"
))

fig_flow.update_layout(
    title="Flow map (origin-destination lines)",
    geo=dict(projection_type="natural earth", showland=True, landcolor="rgb(243,243,243)")
)
fig_flow.show()

## Reflection
- Projection nào phù hợp cho global overview, projection nào phù hợp storytelling?
- Khi nào dùng choropleth, khi nào dùng cartogram (bubble/tile)?
- Flow map có thể gây hiểu lầm gì nếu không chuẩn hoá theo baseline dân số/khoảng cách?